# 🗳️ Encuestas Presidenciales Colombia 2026 — Pipeline de Producción

**Ingesta, armoniza y analiza microdatos de las 12 encuestas oficiales registradas en el CNE.**

> Este notebook ejecuta el paquete `encuestas_lib`: lee los archivos de cada encuestadora (Atlas Intel, GAD3, Invamer, CNC), armoniza nombres de candidatos y demografías, calcula intención de voto agregada con ponderación entre encuestas, y produce un conjunto consolidado de tablas — básicas y avanzadas — listas para análisis o publicación.

Repositorio del paquete: el código vive en el ZIP `encuestas_refactor.zip` que descargaste, dentro de la carpeta `encuestas_lib/`.

---

## ✅ Antes de empezar — checklist

Cumple estos **3 pasos obligatorios** antes de correr cualquier celda:

1. **Subir el paquete y los datos a Google Drive** con la siguiente estructura:

   ```
   MyDrive/Pruebas/encuestas_presidenciales_2026/
   ├── encuestas_lib/         ← código del paquete (del ZIP)
   ├── configs/               ← archivos YAML (del ZIP)
   ├── pyproject.toml         ← (del ZIP)
   ├── requirements.txt       ← (del ZIP)
   └── data/
       └── raw/               ← AQUÍ van los archivos de las encuestas
           ├── 04. CNE-E-DG-2026-000755 - ATLAS INTEL/
           │   └── Atlas Semana E126 Raw Data 010926.xlsx
           ├── 05. CNE-E-DG-2026-001504 - GAD3/
           │   └── 505-221 RCN Enero_4.xlsx
           └── ...  (ver Step 4 para la lista completa)
   ```

2. **Runtime** estándar es suficiente (no necesita GPU). Pero conviene usar **High-RAM** si vas a procesar las 12 encuestas a la vez (44k filas, ~3GB en memoria).

3. **Conoces tu WORKSPACE**: la ruta raíz en Drive que contiene `encuestas_lib/`, `configs/` y `data/raw/`. Por defecto: `/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026`.

---

## 📋 Qué hace cada paso

| Step | Hace |
|------|------|
| 1 | Monta Google Drive |
| 2 | Configuración (ÚNICA celda que editas) |
| 3 | Instala el paquete `encuestas_lib` en modo editable |
| 4 | **Preflight check**: lista los 12 archivos esperados y reporta cuáles encontró |
| 5 | Smoke test: corre 1 sola encuesta para confirmar que todo conecta |
| 6 | Ingestión completa: lee, armoniza y consolida los 12 archivos |
| 7 | Análisis: genera ~25 tablas (básicas + avanzadas) y las exporta a Excel/JSON |
| 8 | Exploración: visualiza los resultados y corre análisis ad-hoc |

---
## ✅ Step 1 — Montar Google Drive

Esto monta tu Drive en `/content/drive/`, permitiendo que el notebook lea archivos de tu cuenta.

**Debes correr esta celda al inicio de cada sesión de Colab.** Aparecerá una ventana pidiendo autorización — sigue el link y pega el código que te dé Google.

In [41]:
from google.colab import drive

drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive montado en /content/drive


---
## ✅ Step 2 — Configuración (la única celda que necesitas editar)

> 💡 **Edita SOLO esta celda.** Todas las demás reutilizan estas variables sin cambios.

### Qué hace cada parámetro

| Parámetro | Para qué sirve |
|---|---|
| `WORKSPACE` | Carpeta raíz en Drive. Debe contener `encuestas_lib/`, `configs/` y `data/raw/`. |
| `WEIGHTING_STRATEGY` | Estrategia para ponderar encuestas entre sí: `uniform`, `sample_size`, `recency_decay`, `inverse_recency_size` (default), o `manual`. Documentado en `configs/weights.yaml`. |
| `FORCE_REINGEST` | `True` reprocesa todos los archivos ignorando el checkpoint parquet. Útil cuando cambiaste el código o los datos. |
| `SKIP_MISSING_FILES` | `True` omite encuestas cuyo archivo no encontraste en Drive. Útil para probar con un subconjunto antes de tenerlas todas. |
| `RUN_SMOKE_TEST` | `True` corre el Step 5 (test rápido con 1 sola encuesta). Recomendado siempre la primera vez. |

In [42]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDITAR AQUÍ — esta es la única celda que requiere cambios
# ══════════════════════════════════════════════════════════════════

# ── Carpeta raíz en tu Drive ──────────────────────────────────────
WORKSPACE = '/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026'

# ── Estrategia de ponderación entre encuestas ─────────────────────
# 'uniform'              — todas las encuestas pesan 1
# 'sample_size'          — peso proporcional al n declarado
# 'recency_decay'        — las más recientes pesan más (half-life 21 días)
# 'inverse_recency_size' — combina n × decaimiento temporal  ⭐ RECOMENDADO
# 'manual'               — pesos hardcoded en configs/weights.yaml
WEIGHTING_STRATEGY = 'inverse_recency_size'

# ── Opciones de ejecución ─────────────────────────────────────────
FORCE_REINGEST = False        # True = ignora checkpoint y reprocesa todo
SKIP_MISSING_FILES = True     # True = omite encuestas sin archivo (útil al empezar)
RUN_SMOKE_TEST = True         # True = corre Step 5 antes de la ingesta completa

# ── Nombres de los archivos de salida ─────────────────────────────
OUTPUT_EXCEL = 'analisis_consolidado.xlsx'
OUTPUT_JSON  = 'analisis_consolidado.json'

print(f'📁 Workspace: {WORKSPACE}')
print(f'⚖️  Estrategia de pesos: {WEIGHTING_STRATEGY}')
print(f'🔁 Forzar reingesta: {FORCE_REINGEST}')
print(f'⏭️  Omitir faltantes: {SKIP_MISSING_FILES}')

📁 Workspace: /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026
⚖️  Estrategia de pesos: inverse_recency_size
🔁 Forzar reingesta: False
⏭️  Omitir faltantes: True


---
## ✅ Step 3 — Instalar el paquete `encuestas_lib`

Instala el paquete en **modo editable** (`pip install -e .`) desde tu Drive. Modo editable significa que si modificas el código fuente en `encuestas_lib/`, los cambios se reflejan al re-ejecutar las celdas sin reinstalar.

> ⚠️ **Solo la PRIMERA vez** que corres esto en una sesión nueva de Colab: después de instalar, ve a `Entorno de ejecución → Reiniciar entorno de ejecución` y vuelve a correr desde el Step 1. Es un requisito de Colab — los paquetes recién instalados no son visibles hasta que reinicies el kernel.

In [43]:
%%time
import subprocess, sys
from pathlib import Path

PKG_ROOT = Path(WORKSPACE)

if not (PKG_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(
        f'❌ No se encontró pyproject.toml en {PKG_ROOT}.\n'
        f'   Verifica que descomprimiste encuestas_refactor.zip en esa carpeta.'
    )

print(f'📦 Instalando desde {PKG_ROOT} (modo editable)…')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(PKG_ROOT), '--quiet'],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print('✅ encuestas_lib instalado en modo editable')
    print('⚠️  Si es la PRIMERA instalación: Entorno → Reiniciar entorno y volver al Step 1')
else:
    print('❌ Falló la instalación:')
    print(result.stderr[-1500:])

📦 Instalando desde /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026 (modo editable)…
✅ encuestas_lib instalado en modo editable
⚠️  Si es la PRIMERA instalación: Entorno → Reiniciar entorno y volver al Step 1
CPU times: user 2.02 ms, sys: 1.55 ms, total: 3.57 ms
Wall time: 13.6 s


---
## ✅ Step 4 — Preflight check: ¿qué archivos están en Drive?

Esta celda **NO procesa nada todavía**. Solo audita: lee el registro de encuestas en `configs/surveys.yaml` y verifica cuáles archivos están en tu Drive.

### 📂 Lista exacta de archivos que el pipeline espera en `data/raw/`

Subcarpetas con sus nombres tal cual el CNE las publica:

| # | Encuestadora | Fecha | Archivo |
|---|---|---|---|
| 1 | Atlas Intel | 2026-01-09 | `04. CNE-E-DG-2026-000755 - ATLAS INTEL/Atlas Semana E126 Raw Data 010926.xlsx` |
| 2 | Atlas Intel | 2026-02-05 | `13. CNE-E-DG-2026-004955 - ATLAS INTEL/Atlas Semana E226 Raw Data 020526_1 (1).csv` |
| 3 | Atlas Intel | 2026-02-28 | `27. CNE-E-DG-2026-008698 - ATLAS INTEL/Base de Dados Atlas Semana 022826_2.xlsx` |
| 4 | Atlas Intel | 2026-03-12 | `29. CNE-E-DG-2026-011140 - ATLAS INTEL/Base de Dados Atlas Semana 031226_1.xlsx` |
| 5 | Atlas Intel | 2026-04-10 | `36. CNE-E-DG-2026-014018 - ATLAS INTEL/Base de Dados Atlas Semana 041026_1.xlsx` |
| 6 | GAD3 | 2026-01-13 | `05. CNE-E-DG-2026-001504 - GAD3/505-221 RCN Enero_4.xlsx` |
| 7 | GAD3 | 2026-02-16 | `23. CNE-E-DG-2026-008418 - GAD3/Anexo IV. B. Microdatos estudio (SPSS).sav` |
| 8 | GAD3 | 2026-03-16 | `31. CNE-E-DG-2026-011911 - GAD3/Anexo IV.A.Microdatos estudio (Excel).xlsx` |
| 9 | Invamer | 2026-02-25 | `19. CNE-E-DG-2026-008198 - INVAMER/3. Data (Regsitros primarios).xlsx` |
| 10 | Invamer | 2026-04-25 | `38. CNE-E-DG-2026-015879 - INVAMER/3. Data (Regsitros primarios).xlsx` |
| 11 | CNC | 2026-02-20 | `25. CNE-E-DG-2026-009065 - CENTRO NACIONAL DE CONSULTORÍA/RevCambioFebrero/data/CC892901_BASE_REVISTA_CAMBIO.sav` |
| 12 | CNC | 2026-03-16 | `34. CNE-E-DG-2026-012431 - CENTRO NACIONAL DE CONSULTORIA/Base de datos/CC893201_POLITICA_REVISTA_CAMBIO_TEXTOS.xlsx` |

> 💡 **Atención a los detalles:**
> - Los nombres y prefijos numéricos (`04.`, `13.`, etc.) deben coincidir **exactamente**. Drive y Linux son case-sensitive.
> - Las dos Invamer (`19.` y `38.`) tienen el archivo con el **mismo nombre** dentro — están en subcarpetas distintas.
> - La CNC del 20 de feb tiene un nivel extra: `RevCambioFebrero/data/...`
> - Para agregar una encuesta nueva: edita `configs/surveys.yaml` y agrega un bloque al final. No requiere tocar código si la encuestadora ya tiene reader.

In [44]:
from encuestas_lib.config import Config, WeightingConfig
from encuestas_lib.pipeline import IngestPipeline
from dataclasses import replace

# Cargar configuración desde configs/ (YAMLs)
config = Config.from_yaml(
    configs_dir=f'{WORKSPACE}/configs',
    root=WORKSPACE,  # data/raw vivirá en {WORKSPACE}/data/raw
)

# Forzar la estrategia de pesos elegida en la celda de configuración
from encuestas_lib.config import WeightingConfig
config = replace(config, weighting=replace(config.weighting, active_strategy=WEIGHTING_STRATEGY))

print(f'✅ Config cargada: {len(config.surveys)} encuestas registradas')
print(f'   Estrategia activa: {config.weighting.active_strategy}')
print(f'   Datos crudos en:  {config.paths.raw_dir}')
print(f'   Outputs en:       {config.paths.outputs_dir}')

# Reporte preflight
ingest = IngestPipeline(config)
report = ingest.preflight()
report

✅ Config cargada: 12 encuestas registradas
   Estrategia activa: inverse_recency_size
   Datos crudos en:  /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/raw
   Outputs en:       /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs


Preflight: 12/12 archivos disponibles en /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/raw

ℹ 1 archivo(s) resueltos vía normalización Unicode (NFC↔NFD). Drive desde Mac suele causar esto.

,id,encuestadora,fecha,reader,path_relativo,existe,tamano_mb,unicode_fix
0,atlas_2026_01_09,Atlas Intel,2026-01-09,atlas,04. CNE-E-DG-2026-000755 - ATLAS INTEL/Atlas S...,True,0.58,False
1,atlas_2026_02_05,Atlas Intel,2026-02-05,atlas,13. CNE-E-DG-2026-004955 - ATLAS INTEL/Atlas S...,True,6.04,False
2,atlas_2026_02_28,Atlas Intel,2026-02-28,atlas,27. CNE-E-DG-2026-008698 - ATLAS INTEL/Base de...,True,1.02,False
3,atlas_2026_03_12,Atlas Intel,2026-03-12,atlas,29. CNE-E-DG-2026-011140 - ATLAS INTEL/Base de...,True,0.63,False
4,atlas_2026_04_10,Atlas Intel,2026-04-10,atlas,36. CNE-E-DG-2026-014018 - ATLAS INTEL/Base de...,True,0.45,False
5,gad3_2026_01_13,GAD3,2026-01-13,gad3_excel_v1,05. CNE-E-DG-2026-001504 - GAD3/505-221 RCN En...,True,0.24,False
6,gad3_2026_02_16,GAD3,2026-02-16,gad3_sav,23. CNE-E-DG-2026-008418 - GAD3/Anexo IV. B. M...,True,0.40,False
7,gad3_2026_03_16,GAD3,2026-03-16,gad3_excel_v2,31. CNE-E-DG-2026-011911 - GAD3/Anexo IV.A.Mic...,True,0.26,False
8,invamer_2026_02_25,Invamer,2026-02-25,invamer,19. CNE-E-DG-2026-008198 - INVAMER/3. Data (Re...,True,2.55,False
9,invamer_2026_04_25,Invamer,2026-04-25,invamer,38. CNE-E-DG-2026-015879 - INVAMER/3. Data (Re...,True,2.27,False


---
## ✅ Step 5 — Smoke test (1 sola encuesta)

**No saltes este paso.** Lee la primera encuesta disponible en menos de 30 segundos y confirma que:

- El reader correspondiente puede leer el archivo (formato, encoding correctos)
- La armonización de nombres de candidatos funciona
- El schema canónico de salida está completo

Si el smoke test falla, **el problema lo encontrarás en 30 segundos** en vez de gastar 5 minutos en la ingesta completa.

> Pon `RUN_SMOKE_TEST = False` en el Step 2 para saltarlo (no recomendado la primera vez).

In [45]:
if RUN_SMOKE_TEST:
    from encuestas_lib.readers import get_reader_class
    from encuestas_lib.harmonization import build_harmonizer
    import time

    # Encuestas disponibles según el preflight
    disponibles = [s for s in config.surveys if s.path.exists()]
    if not disponibles:
        raise FileNotFoundError(
            '⚠️  Ninguna encuesta encontrada. Sube al menos un archivo a data/raw/ '
            'siguiendo la tabla del Step 4.'
        )

    primer = disponibles[0]
    print(f'🧪 Smoke test con: {primer.id} ({primer.path.name})')

    harmonizer = build_harmonizer(config.candidates_raw, config.special_categories_raw)
    ReaderCls = get_reader_class(primer.reader)

    t0 = time.time()
    df_smoke = ReaderCls(primer, harmonizer).read()
    elapsed = time.time() - t0

    print(f'\n✅ Smoke test PASÓ en {elapsed:.1f}s')
    print(f'   Filas leídas:          {len(df_smoke):,}')
    print(f'   Columnas canónicas:    {len(df_smoke.columns)}')
    print(f'   Candidatos detectados: {df_smoke["primera_vuelta"].nunique()}')
    print(f'\nTop 5 candidatos en esta encuesta:')
    print(df_smoke["primera_vuelta"].value_counts().head().to_string())
else:
    print('⏭️  Smoke test omitido (RUN_SMOKE_TEST=False)')

🧪 Smoke test con: atlas_2026_01_09 (Atlas Semana E126 Raw Data 010926.xlsx)

✅ Smoke test PASÓ en 5.4s
   Filas leídas:          4,520
   Columnas canónicas:    19
   Candidatos detectados: 19

Top 5 candidatos en esta encuesta:
primera_vuelta
Iván Cepeda                 2071
Abelardo de la Espriella    1290
Sergio Fajardo               327
Paloma Valencia              198
Juan Carlos Pinzón           112


---
## ✅ Step 6 — Ingestión completa

Lee todos los archivos disponibles, los armoniza al schema canónico (mismas columnas y nombres de candidato en todas las encuestas), y los concatena en un único DataFrame.

El resultado se cachea en `data/processed/encuestas_concatenadas.parquet`. Las siguientes ejecuciones leen el parquet directamente (segundos en vez de minutos), salvo que pongas `FORCE_REINGEST=True`.

**Tiempo esperado**: ~2-4 minutos para las 12 encuestas en Colab estándar.

In [46]:
%%time
df = ingest.run(forzar=FORCE_REINGEST, skip_missing=SKIP_MISSING_FILES)

print(f'\n📊 DataFrame consolidado:')
print(f'   Filas:        {len(df):,}')
print(f'   Columnas:     {len(df.columns)}')
print(f'   Encuestas:    {df[["encuestadora", "fecha"]].drop_duplicates().shape[0]}')
print(f'   Rango fechas: {df["fecha"].min()} a {df["fecha"].max()}')

♻  Cache hit: encuestas_concatenadas.parquet (44,517 filas) — fingerprint ✓


📊 DataFrame consolidado:
   Filas:        44,517
   Columnas:     32
   Encuestas:    12
   Rango fechas: 2026-01-09 00:00:00 a 2026-04-25 00:00:00
CPU times: user 251 ms, sys: 47.7 ms, total: 298 ms
Wall time: 1.96 s


In [47]:
# Filas por encuestadora
df.groupby('encuestadora').size().sort_values(ascending=False).to_frame('n_filas')

,n_filas
encuestadora,
Atlas Intel,26194
Invamer,7602
CNC,5841
GAD3,4880


---
## ✅ Step 7 — Análisis y exportación

Corre el `AnalysisPipeline`: genera **~25 tablas** (entre básicas y avanzadas) y las exporta como:

- **Excel multi-hoja** en `data/outputs/analisis_consolidado.xlsx` (una hoja por tabla)
- **JSON con metadatos** en `data/outputs/analisis_consolidado.json` (timestamp, hash, todas las tablas)

Al final imprime un reporte de validación forense: cuántas tablas cierran a 100% (auditoría automática).

### Tablas generadas

**Básicas (reproducen el repo original):**
- `primera_vuelta_total` — intención de voto agregada
- `voto_por_region`, `voto_por_edad`, `voto_por_genero`
- `aprobacion_vs_voto`, `voto_vs_aprobacion`
- `sesgo_genero`, `sesgo_edad`, `sesgo_region` — house effects
- `indecisos_total`, `indecisos_region`, `indecisos_edad_grupo`, `indecisos_sexo`

**Avanzadas (NUEVAS):**
- `trend_primera_vuelta` — serie temporal con suavizado por ventana móvil
- `coalicion_aprobacion` — voto × aprobación Petro
- `volatilidad_encuestadora` — desviación intra-pollster
- `indecisos_perfil` — tasa de indecisión por dimensión
- `moe_*` — margen de error con n efectivo de Kish (para top-3 candidatos)
- `transfer_sv_*` — transferencia PV → SV por matchup
- `techo_potencial_*` — espacio de crecimiento PV → SV
- `_validacion_cierres` — auditoría forense de cierres a 100

In [48]:
%%time
from encuestas_lib.pipeline import AnalysisPipeline

tablas = AnalysisPipeline(config).run(
    df,
    excel_name=OUTPUT_EXCEL,
    json_name=OUTPUT_JSON,
    validar=True,
)

print(f'\n📋 Total de tablas generadas: {len(tablas)}')

ℹ Filtro candidatos_principales activo: 41,029 de 44,517 filas conservadas (92.2%). Principales: 13 · Especiales: 
6.

ℹ Remap indecisos: 3,123 votos no-principales → 'Otro candidato' (necesario para paridad con repo original).

Generando tablas básicas…

Generando tablas avanzadas…

   Validación de cierres a 100%   
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Métrica                ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Filas validadas        │ 46    │
│ OK                     │ 46    │
│ Fallidas               │ 0     │
│ Desviación máxima (pp) │ 0.020 │
└────────────────────────┴───────┘

✓ Excel: /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs/analisis_consolidado.xlsx

✓ JSON:  /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs/analisis_consolidado.json


📋 Total de tablas generadas: 41
CPU times: user 7.66 s, sys: 133 ms, total: 7.79 s
Wall time: 8.11 s


---
## ✅ Step 8 — Exploración de resultados

Las tablas viven en el dict `tablas` (en memoria) y también en disco (Excel + JSON).

### 8.1 — Intención de voto agregada en primera vuelta

In [49]:
tablas['primera_vuelta_total'].sort_values('valor', ascending=False)

,primera_vuelta,valor
0,Iván Cepeda,29.64
1,Abelardo de la Espriella,19.17
2,Paloma Valencia,12.29
3,Ninguno,11.0
4,NS/NR,7.71
5,Sergio Fajardo,3.93
6,Voto en blanco,3.68
7,No votaría,3.55
8,No sé,2.82
9,Claudia López,2.72


### 8.2 — Tendencia temporal (top-5 candidatos)

Esta tabla está en formato long: `[fecha, primera_vuelta, valor_punto, valor_suavizado]`. Cada candidato tiene una serie temporal con dos columnas: el punto observado por encuesta y la media móvil ponderada por ventana de 14 días.

In [50]:
trend = tablas['trend_primera_vuelta']
top5 = (tablas['primera_vuelta_total']
        .nlargest(5, 'valor')['primera_vuelta'].tolist())
trend[trend['primera_vuelta'].isin(top5)].head(30)

,fecha,valor_punto,valor_suavizado,primera_vuelta
0,2026-01-09,27.986098,27.986098,Abelardo de la Espriella
1,2026-01-13,21.921506,24.953802,Abelardo de la Espriella
2,2026-02-05,28.398659,28.398659,Abelardo de la Espriella
3,2026-02-16,26.312725,27.355692,Abelardo de la Espriella
4,2026-02-20,16.718634,21.515679,Abelardo de la Espriella
5,2026-02-25,17.020964,20.017441,Abelardo de la Espriella
6,2026-02-28,31.912015,22.991084,Abelardo de la Espriella
7,2026-03-12,26.807549,29.359782,Abelardo de la Espriella
8,2026-03-16,15.457312,21.132431,Abelardo de la Espriella
9,2026-03-16,20.594332,20.953064,Abelardo de la Espriella


### 8.3 — Validación forense de cierres a 100%

Cada tabla que **debe** sumar 100% (intención de voto, demografías, etc.) es auditada automáticamente. Tolerancia: 0.5 puntos porcentuales por defecto.

In [51]:
tablas['_validacion_cierres']

,tabla,grupo,suma_observada,desviacion,ok
0,primera_vuelta_total,<total>,100.01,0.01,True
1,voto_por_region,fila_0,100.01,0.01,True
2,voto_por_region,fila_1,99.99,0.01,True
3,voto_por_region,fila_2,100.00,0.00,True
4,voto_por_region,fila_3,100.00,0.00,True
5,voto_por_region,fila_4,100.01,0.01,True
6,voto_por_region,fila_5,100.00,0.00,True
7,voto_por_region,fila_6,99.99,0.01,True
8,voto_por_region,fila_7,100.01,0.01,True
9,voto_por_region,fila_8,99.99,0.01,True


### 8.4 — Margen de error efectivo (IC95% con n de Kish)

Para cada encuesta, calcula el margen de error a 95% **corrigiendo el n nominal por la dispersión de los factores de expansión**. Si los factores son muy heterogéneos, el n efectivo es bastante menor que el n declarado — y eso amplía el IC.

In [52]:
# Para el candidato #1 (líder)
lider_key = sorted(
    [k for k in tablas if k.startswith('moe_')]
)[0]
tablas[lider_key]

,encuestadora,fecha,n_nominal,n_efectivo,p_estimado_pct,moe_pp,ic95_lo,ic95_hi
0,Atlas Intel,2026-01-09,4520,359.6,26.50,4.56,21.94,31.06
1,GAD3,2026-01-13,1207,621.4,30.26,3.61,26.65,33.88
2,Atlas Intel,2026-02-05,7298,1244.1,27.83,2.49,25.34,30.32
3,GAD3,2026-02-16,2108,1510.5,34.15,2.39,31.76,36.54
4,CNC,2026-02-20,3684,1909.9,35.40,2.14,33.25,37.54
5,Invamer,2026-02-25,3800,3082.0,33.32,1.66,31.66,34.99
6,Atlas Intel,2026-02-28,6468,610.3,34.01,3.76,30.25,37.76
7,Atlas Intel,2026-03-12,4291,280.9,34.95,5.58,29.38,40.53
8,CNC,2026-03-16,2157,1240.8,34.47,2.64,31.83,37.12
9,GAD3,2026-03-16,1200,1076.2,35.25,2.85,32.39,38.10


### 8.5 — House effects (sesgo de cada encuestadora vs el promedio del resto)

Si una encuestadora reporta sistemáticamente más mujeres que el resto, esto lo muestra. Útil para auditar la metodología.

In [53]:
tablas['sesgo_region'].head(20)

,encuestadora,variable,categoria,peso_encuestadora,peso_promedio_otras,sesgo_rel_pp
0,Atlas Intel,region,Central,32.67,19.12,13.55
1,Atlas Intel,region,Bogotá,21.46,11.06,10.40
2,Atlas Intel,region,Caribe,22.32,15.54,6.78
3,Atlas Intel,region,Pacífico,17.33,11.19,6.14
4,Atlas Intel,region,Amazonía - Orinoquía,6.21,2.69,3.52
5,CNC,region,Caribe,21.97,14.03,7.94
6,CNC,region,Eje Cafetero,18.64,11.95,6.69
7,CNC,region,Pacífico,15.96,10.50,5.46
8,CNC,region,Centro - Sur - Amazonía,6.40,3.72,2.68
9,CNC,region,Bogotá,15.82,13.31,2.51


### 8.6 — Volatilidad intra-encuestadora

Desviación estándar entre mediciones de la misma encuestadora. Volatilidad alta puede ser ruido metodológico, o captura legítima de movimiento — no se puede saber sin más contexto.

In [54]:
tablas['volatilidad_encuestadora'].head(20)

,encuestadora,primera_vuelta,n_mediciones,media_pct,desv_std_pp,min_pct,max_pct,rango_pp
43,GAD3,Abelardo de la Espriella,3,22.942854,2.992886,20.594332,26.312725,5.72
0,Atlas Intel,Abelardo de la Espriella,5,28.46791,2.023308,26.807549,31.912015,5.1
21,CNC,Abelardo de la Espriella,2,16.087973,0.891889,15.457312,16.718634,1.26
63,Invamer,Abelardo de la Espriella,2,17.114874,0.132808,17.020964,17.208783,0.19
44,GAD3,Aníbal Gaviria,2,0.757416,0.505557,0.399933,1.114899,0.71
1,Atlas Intel,Aníbal Gaviria,3,1.106905,0.208798,0.975251,1.347652,0.37
23,CNC,Carlos Caicedo,2,0.656469,0.751962,0.124751,1.188186,1.06
45,GAD3,Carlos Caicedo,2,0.202458,0.077685,0.147527,0.257389,0.11
2,Atlas Intel,Clara López,2,0.54099,0.762351,0.001926,1.080053,1.08
65,Invamer,Claudia López,2,7.572765,5.259318,3.853865,11.291664,7.44


---
## 🛠️ Step 9 — Análisis ad-hoc (EXTRAS)

Las funciones del módulo `encuestas_lib.analysis` son puras: reciben el DataFrame y devuelven un DataFrame. Útiles para preguntas que no están en el pipeline estándar.

### 9.1 — Transferencia de voto PV → SV para una matchup específica

In [55]:
from encuestas_lib.analysis import transferencia_pv_sv, resolve_weights, sv_columns

pesos = resolve_weights(config.weighting, config.surveys)
sv_disponibles = sv_columns(df)
print('Matchups SV disponibles:', sv_disponibles)

# Ejemplo: transferencia para la primera matchup
if sv_disponibles:
    transferencia_pv_sv(df, sv_disponibles[0], pesos)

Matchups SV disponibles: ['sv_barreras_vs_cepeda', 'sv_cardenas_vs_cepeda', 'sv_cepeda_vs_claudia', 'sv_cepeda_vs_davila', 'sv_cepeda_vs_espriella', 'sv_cepeda_vs_fajardo', 'sv_cepeda_vs_galan', 'sv_cepeda_vs_gaviria', 'sv_cepeda_vs_lizcano', 'sv_cepeda_vs_murillo', 'sv_cepeda_vs_pinzon', 'sv_cepeda_vs_uribe_londono', 'sv_cepeda_vs_valencia', 'sv_espriella_vs_fajardo', 'sv_espriella_vs_valencia', 'sv_fajardo_vs_pinzon', 'sv_fajardo_vs_valencia']


### 9.2 — Perfil de indecisos por dimensión

In [56]:
from encuestas_lib.analysis import indecisos_perfil
from encuestas_lib.harmonization import build_harmonizer

harm = build_harmonizer(config.candidates_raw, config.special_categories_raw)
indecisos_perfil(df, harm.vigentes())

,dimension,categoria,tasa_indecisos_pct,n
17,aprobacion_petro,NS/NR,48.58,701.0
18,aprobacion_petro,Regular,26.26,6258.0
16,aprobacion_petro,Desaprueba,17.69,16962.0
15,aprobacion_petro,Aprueba,12.94,15716.0
9,edad_grupo,18-34,17.72,13048.0
11,edad_grupo,55+,17.21,18533.0
10,edad_grupo,35-54,16.33,12569.0
14,genero,Mujer,20.32,20603.0
13,genero,Hombre,13.59,23547.0
12,genero,B_SEXO,NaN,2.0


### 9.3 — Margen de error para cualquier candidato

Cambia el nombre canónico para inspeccionar el IC de otro candidato.

In [57]:
from encuestas_lib.analysis import margen_error_efectivo

margen_error_efectivo(df, 'Iván Cepeda')

,encuestadora,fecha,n_nominal,n_efectivo,p_estimado_pct,moe_pp,ic95_lo,ic95_hi
0,Atlas Intel,2026-01-09,4520,359.6,26.50,4.56,21.94,31.06
1,GAD3,2026-01-13,1207,621.4,30.26,3.61,26.65,33.88
2,Atlas Intel,2026-02-05,7298,1244.1,27.83,2.49,25.34,30.32
3,GAD3,2026-02-16,2108,1510.5,34.15,2.39,31.76,36.54
4,CNC,2026-02-20,3684,1909.9,35.40,2.14,33.25,37.54
5,Invamer,2026-02-25,3800,3082.0,33.32,1.66,31.66,34.99
6,Atlas Intel,2026-02-28,6468,610.3,34.01,3.76,30.25,37.76
7,Atlas Intel,2026-03-12,4291,280.9,34.95,5.58,29.38,40.53
8,CNC,2026-03-16,2157,1240.8,34.47,2.64,31.83,37.12
9,GAD3,2026-03-16,1200,1076.2,35.25,2.85,32.39,38.10


---
## ✅ Step 10 — Verificación de outputs

Confirma que los archivos finales se generaron correctamente en Drive.

In [58]:
from pathlib import Path

out_dir = Path(WORKSPACE) / 'data' / 'outputs'
for f in sorted(out_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name:<40} {size_mb:>6.2f} MB')

print(f'\n📂 Ubicación: {out_dir}')
print('   Estos archivos están en tu Drive y puedes descargarlos o compartirlos.')

   analisis_consolidado.json                  0.37 MB
   analisis_consolidado.xlsx                  0.09 MB
   graficas                                   0.00 MB

📂 Ubicación: /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs
   Estos archivos están en tu Drive y puedes descargarlos o compartirlos.


---
## ✅ Step 11 — Dashboard interactivo (Plotly → HTML)

Genera **todas las gráficas del PDF de La Silla Vacía** como visualizaciones Plotly interactivas (zoom, hover, descarga PNG) y las combina en un único archivo `dashboard_interactivo.html` autosuficiente.

| Celda | Contenido |
|---|---|
| 11.0 | Setup: Plotly, paleta LSV, helpers |
| 11.1 | Tendencia temporal top-5 candidatos |
| 11.2 | Sankey PV → SV (dos matchups) |
| 11.3 | Trasvase de la derecha (serie temporal) |
| 11.4 | Perfil de indecisos: edad · género · región |
| 11.5 | Barras 100% apiladas: edad · género · región |
| 11.6 | Sesgo demográfico por encuestadora |
| 11.7 | Petrismo — votantes de Cepeda que aprueban a Petro |
| 11.8 | Composición por género de cada candidato |
| 11.9 | Primera vuelta total + indecisos desglosados |
| 11.10 | **Exportar** todas las figuras a `dashboard_interactivo.html` |

### 11.0 — Setup: Plotly, paleta LSV, helpers

In [59]:
# ══════════════════════════════════════════════════════════════════════
#  Step 11 · Setup — Plotly interactivo
# ══════════════════════════════════════════════════════════════════════
#
# Esta celda registra el template visual ``lsv`` y prepara los handles
# globales que las demás celdas de Step 11 esperan en el notebook.
# La lógica vive en ``encuestas_lib.viz`` para que sea testable.
# ──────────────────────────────────────────────────────────────────────
from collections import OrderedDict
from pathlib import Path

import plotly.graph_objects as go

from encuestas_lib.viz import (
    INDECISOS_CATS, LAYOUT_BASE, PALETA, c, hex_to_rgba, register_template,
)

# Activar template y dejarlo como default (resuelve el TypeError 'yaxis duplicado')
register_template()

# ── Helpers de candidatos (derivados del dict ``tablas`` que produce el pipeline) ──
TOP5 = (
    tablas["primera_vuelta_total"]
    .nlargest(5, "valor")["primera_vuelta"]
    .tolist()
)
ALL_CANDS = [
    cd for cd in tablas["primera_vuelta_total"]["primera_vuelta"].tolist()
    if cd not in INDECISOS_CATS
]
ORDEN_PV = ALL_CANDS + [
    cd for cd in tablas["primera_vuelta_total"]["primera_vuelta"].tolist()
    if cd in INDECISOS_CATS
]

# ── Directorio de salida de gráficas ──
FIG_DIR = Path(WORKSPACE) / "data" / "outputs" / "graficas"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Dict ordenado de figuras (alimenta el dashboard HTML al final)
FIGURAS: "OrderedDict[str, go.Figure]" = OrderedDict()

print("✅ Setup Plotly listo")
print(f"   Top 5 candidatos : {TOP5}")
print(f"   Guardando en     : {FIG_DIR}")


✅ Setup Plotly listo
   Top 5 candidatos : ['Iván Cepeda', 'Abelardo de la Espriella', 'Paloma Valencia', 'Ninguno', 'NS/NR']
   Guardando en     : /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs/graficas


### 11.1 — Tendencia temporal

Intención de voto del top-5 con ribbon de incertidumbre. Hover para ver valor exacto por fecha.

In [60]:
# ── 11.1 Tendencia temporal top-5 ───────────────────────────────────
from encuestas_lib.viz.charts.step11 import chart_tendencia_temporal

fig = chart_tendencia_temporal(tablas, top=5)
FIGURAS["01_tendencia"] = fig
fig.show()


### 11.2 — Sankey: movimiento de votos PV → SV

Cada flujo = (% en PV) × (% de transferencia a SV). Hover sobre nodos y enlaces para ver valores. Arrastra nodos para reorganizar.

In [61]:
# ── 11.2 Sankey PV → SV (dos matchups) ─────────────────────────────
from encuestas_lib.viz.charts.step11 import chart_sankey_pv_sv

fig_sv1 = chart_sankey_pv_sv(
    tablas,
    tabla_key="transfer_sv_cepeda_vs_valencia",
    sv_col="sv_cepeda_vs_valencia",
    titulo="Movimiento de votos PV → SV · Escenario Cepeda vs Valencia",
    subtitle=(
        "Cada flujo = (% en PV) × (% transferencia a SV). "
        "Hover para ver valores exactos."
    ),
)
FIGURAS["02a_sankey_cepeda_valencia"] = fig_sv1
fig_sv1.show()

fig_sv2 = chart_sankey_pv_sv(
    tablas,
    tabla_key="transfer_sv_cepeda_vs_espriella",
    sv_col="sv_cepeda_vs_espriella",
    titulo="Movimiento de votos PV → SV · Escenario Cepeda vs Espriella",
    subtitle=(
        "Cada flujo = (% en PV) × (% transferencia a SV). "
        "Hover para ver valores exactos."
    ),
)
FIGURAS["02b_sankey_cepeda_espriella"] = fig_sv2
fig_sv2.show()


/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/encuestas_lib/viz/charts/step11.py:138: RuntimeWarning:

Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.



/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/encuestas_lib/viz/charts/step11.py:138: RuntimeWarning:

Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.



### 11.3 — Trasvase de la derecha (serie temporal)

Por encuesta y fecha: % de votantes de Abelardo que irían a Paloma (y viceversa), excluyendo indecisos de segunda vuelta.

In [62]:
# ── 11.3 Trasvase de la derecha (serie temporal) ─────────────────────
from encuestas_lib.viz.charts.step11 import chart_trasvase_derecha

fig_tr = chart_trasvase_derecha(df, tablas)
FIGURAS["03_trasvase"] = fig_tr
fig_tr.show()


### 11.4 — Perfil de los indecisos

In [63]:
# ── 11.4 Perfil de indecisos ─────────────────────────────────────────
from encuestas_lib.viz.charts.step11 import chart_perfil_indecisos

fig_ind = chart_perfil_indecisos(tablas)
FIGURAS["04_indecisos"] = fig_ind
fig_ind.show()


### 11.5 — Barras 100% apiladas

Incluyendo e ignorando indecisos. Hover para ver candidato, grupo demográfico y valor exacto.

In [64]:
# ── 11.5 Barras 100% apiladas (edad, género, región) ─────────────────
from encuestas_lib.viz.charts.step11 import chart_stacked_bar

fig_edad = chart_stacked_bar(
    tablas, tabla_key="voto_por_edad", dim_col="edad_grupo",
    titulo="Intención de voto por edad",
    subtitle="Barras 100% apiladas · Grupos etarios",
    esconder_indecisos=False, height=380,
)
FIGURAS["05a_voto_edad"] = fig_edad
fig_edad.show()

fig_genero = chart_stacked_bar(
    tablas, tabla_key="voto_por_genero", dim_col="sexo",
    titulo="Intención de voto por género",
    subtitle="Barras 100% apiladas",
    esconder_indecisos=False, height=320,
)
FIGURAS["05b_voto_genero"] = fig_genero
fig_genero.show()

fig_region = chart_stacked_bar(
    tablas, tabla_key="voto_por_region", dim_col="region",
    titulo="Intención de voto por región",
    subtitle="Barras 100% apiladas · 9 regiones",
    esconder_indecisos=False, height=520,
)
FIGURAS["05c_voto_region"] = fig_region
fig_region.show()


### 11.6 — Sesgo demográfico por encuestadora

In [65]:
# ── 11.6 Sesgo demográfico por encuestadora ────────────────────────────
from encuestas_lib.viz.charts.step11 import chart_sesgo_demografico

fig_se1 = chart_sesgo_demografico(
    tablas, tabla_key="sesgo_edad",
    titulo="Sesgo demográfico por encuestadora — Edad",
    subtitle="+pp = sobreestima ese grupo etario · −pp = subestima",
)
FIGURAS["06a_sesgo_edad"] = fig_se1
fig_se1.show()

fig_se2 = chart_sesgo_demografico(
    tablas, tabla_key="sesgo_genero",
    titulo="Sesgo demográfico por encuestadora — Género",
    subtitle="+pp = sobreestima ese género · −pp = subestima",
)
FIGURAS["06b_sesgo_genero"] = fig_se2
fig_se2.show()


### 11.7 — Petrismo

In [66]:
# ── 11.7 Petrismo: votantes de Cepeda que aprueban a Petro ───────────
from encuestas_lib.viz.charts.step11 import chart_petrismo_cepeda

fig_petro = chart_petrismo_cepeda(tablas, candidato="Iván Cepeda")
FIGURAS["07_petro"] = fig_petro
fig_petro.show()


### 11.8 — Composición por género de cada candidato

In [67]:
# ── 11.8 Composición por género de cada candidato ─────────────────────
from encuestas_lib.viz.charts.step11 import chart_composicion_genero

fig_gc = chart_composicion_genero(tablas)
FIGURAS["08_genero_composicion"] = fig_gc
fig_gc.show()


### 11.9 — Primera vuelta total

In [68]:
# ── 11.9 Primera vuelta total + indecisos ────────────────────────────
from encuestas_lib.viz.charts.step11 import chart_primera_vuelta_total

fig_pv = chart_primera_vuelta_total(tablas)
FIGURAS["09_pv_total"] = fig_pv
fig_pv.show()


### 11.10 — Exportar dashboard HTML

Combina **todas las figuras** en un único `dashboard_interactivo.html` con sidebar de navegación, diseño responsive y Plotly cargado desde CDN.

In [69]:
# ── 11.10 Exportar dashboard HTML interactivo ─────────────────────────
from encuestas_lib.viz.dashboard import SECCIONES_STEP11, export_dashboard

# En esta celda exportamos solo Step 11 — la celda 12.9 añade Step 12 al
# mismo archivo si está disponible.
out_path = export_dashboard(
    figuras=FIGURAS,
    out_path=FIG_DIR / "dashboard_interactivo.html",
    secciones_step11=SECCIONES_STEP11,
    secciones_step12=(),  # se completa en Step 12
)
size_mb = out_path.stat().st_size / 1e6

print("\n✅ Dashboard exportado correctamente")
print(f"   📄 Archivo : {out_path}")
print(f"   📦 Tamaño  : {size_mb:.2f} MB")
print(f"   📊 Gráficas: {len(FIGURAS)}")
print("\n💡 Para descargar: clic derecho en el archivo en el panel de Drive → Descargar")



✅ Dashboard exportado correctamente
   📄 Archivo : /content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026/data/outputs/graficas/dashboard_interactivo.html
   📦 Tamaño  : 0.17 MB
   📊 Gráficas: 13

💡 Para descargar: clic derecho en el archivo en el panel de Drive → Descargar


---
## ✅ Step 12 — Análisis predictivo avanzado

Análisis cuantitativo adicional extraído de los microdatos y del documento forense **A01 — Consolidación Forense de Predicción Electoral Colombia 2026**.

| Celda | Análisis |
|---|---|
| 12.0 | Setup: importación del módulo `electoral.py`, parámetros del doc. forense |
| 12.1 | Transferencia de voto: Fajardo, Claudia López, Botero y Barreras en ambos matchups |
| 12.2 | Simulación Monte Carlo de segunda vuelta (20 000 iteraciones × 2 escenarios) |
| 12.3 | Análisis de sensibilidad: 3 swing factors críticos |
| 12.4 | Comparativo Polymarket vs encuestas vs modelo de transferencia |
| 12.5 | Techo de rechazo por candidato (doc. forense + estimación propia) |
| 12.6 | Geografía del petrismo: voto Cepeda y concentración de indecisos por región |
| 12.7 | Ruptura generacional: voto joven y abstención diferencial |
| 12.8 | Panel ejecutivo: probabilidades consolidadas doc. forense vs modelo MC |
| 12.9 | Exportar Step 12 al `dashboard_interactivo.html` |

> **Módulo reutilizable**: las funciones de `encuestas_lib.analysis.electoral` están completamente testeadas y tipadas. Para agregar un nuevo escenario de segunda vuelta, define los pesos PV y la matriz de transferencia con rangos (min, max) y llama a `simular_segunda_vuelta()`.

### 12.0 — Setup: módulo electoral, parámetros del documento forense

In [70]:
# ══════════════════════════════════════════════════════════════════════
#  Step 12 · Setup — Análisis predictivo avanzado
# ══════════════════════════════════════════════════════════════════════
from encuestas_lib.analysis.electoral import (
    calcular_techo_rechazo, resumen_escenarios_2v, sensibilidad_2v,
    simular_segunda_vuelta, trasvase_candidato,
)
from encuestas_lib.viz.charts.step12 import MATRIZ_A, MATRIZ_B, PESOS_PV_DOC

# El template ``lsv`` ya fue registrado en el Setup de Step 11.
# Si esta celda corre primero, lo registramos de forma idempotente:
from encuestas_lib.viz import register_template
register_template()

FIGURAS12 = {}
print("✅ Step 12 setup listo — módulo electoral importado")
print(f"   Pesos PV base: {PESOS_PV_DOC}")


✅ Step 12 setup listo — módulo electoral importado
   Pesos PV base: {'Iván Cepeda': 38.0, 'Abelardo de la Espriella': 25.0, 'Paloma Valencia': 19.0, 'Fajardo+López': 5.0, 'Otros': 4.0, 'Blanco/Nulo': 9.0}


### 12.1 — Transferencia de voto del centro

Fajardo, Claudia López, Botero y Barreras: ¿adónde van sus votos en cada matchup de segunda vuelta? Benchmark del documento forense: 35–55% hacia Cepeda.

In [71]:
# ── 12.1 Transferencia de voto: Fajardo y Claudia López ─────────────
# Parámetros centralizados en ``encuestas_lib.viz.charts.step12``.
# Para tunearlos: editar SOLO ese módulo.
from encuestas_lib.viz.charts.step12 import CANDS_CENTRO_DOC, chart_trasvase_centro

fig_tc1 = chart_trasvase_centro(
    tablas,
    sv_col_key="transfer_sv_cepeda_vs_espriella",
    titulo=(
        "Trasvase de votos del centro — Escenario Cepeda vs Espriella<br>"
        "¿A quién van los votantes de Fajardo, Claudia, Botero y Barreras?"
    ),
    candidato_a="Iván Cepeda",
    cands_centro=CANDS_CENTRO_DOC,
)
FIGURAS12["12_01a_trasvase_centro_escA"] = fig_tc1
fig_tc1.show()

fig_tc2 = chart_trasvase_centro(
    tablas,
    sv_col_key="transfer_sv_cepeda_vs_valencia",
    titulo=(
        "Trasvase de votos del centro — Escenario Cepeda vs Valencia<br>"
        "¿A quién van los votantes de Fajardo, Claudia, Botero y Barreras?"
    ),
    candidato_a="Iván Cepeda",
    cands_centro=CANDS_CENTRO_DOC,
)
FIGURAS12["12_01b_trasvase_centro_escB"] = fig_tc2
fig_tc2.show()


### 12.2 — Simulación Monte Carlo de segunda vuelta

Cada iteración muestrea las tasas de transferencia dentro de los rangos de incertidumbre del documento forense (Tablas 7 y 8 del A01) y calcula el resultado de la segunda vuelta.

In [72]:
# ── 12.2 Simulación Monte Carlo de segunda vuelta ────────────────────
# Parámetros centralizados en ``encuestas_lib.viz.charts.step12.MC_PARAMS_DOC``.
# Para cambiar n_iter o seed: editar SOLO ese módulo.
from encuestas_lib.viz.charts.step12 import MC_PARAMS_DOC, chart_monte_carlo

print(
    f"⏳ Corriendo simulaciones Monte Carlo "
    f"({MC_PARAMS_DOC.n_iter:,} iter. × 2 escenarios, seed={MC_PARAMS_DOC.seed})…"
)

res_A, df_iters_A = simular_segunda_vuelta(
    PESOS_PV_DOC, MATRIZ_A,
    candidato_a="Iván Cepeda", candidato_b="Abelardo de la Espriella",
    n_iter=MC_PARAMS_DOC.n_iter, seed=MC_PARAMS_DOC.seed,
)
res_B, df_iters_B = simular_segunda_vuelta(
    PESOS_PV_DOC, MATRIZ_B,
    candidato_a="Iván Cepeda", candidato_b="Paloma Valencia",
    n_iter=MC_PARAMS_DOC.n_iter, seed=MC_PARAMS_DOC.seed,
)

print(f"\n📊 Escenario A — {res_A.candidato_a} vs {res_A.candidato_b}")
print(f"   Cepeda    : {res_A.media_a:.1f}%  "
      f"IC80: {res_A.ic80_a[0]:.1f}–{res_A.ic80_a[1]:.1f}%")
print(f"   Espriella : {res_A.media_b:.1f}%  "
      f"IC80: {res_A.ic80_b[0]:.1f}–{res_A.ic80_b[1]:.1f}%")
print(f"   P(Cepeda gana)    : {res_A.prob_a_gana*100:.1f}%")
print(f"   P(Espriella gana) : {res_A.prob_b_gana*100:.1f}%")
print(f"   P(Empate técnico <2pp): {res_A.prob_empate_tecnico*100:.1f}%")

print(f"\n📊 Escenario B — {res_B.candidato_a} vs {res_B.candidato_b}")
print(f"   Cepeda  : {res_B.media_a:.1f}%  "
      f"IC80: {res_B.ic80_a[0]:.1f}–{res_B.ic80_a[1]:.1f}%")
print(f"   Valencia: {res_B.media_b:.1f}%  "
      f"IC80: {res_B.ic80_b[0]:.1f}–{res_B.ic80_b[1]:.1f}%")
print(f"   P(Cepeda gana)   : {res_B.prob_a_gana*100:.1f}%")
print(f"   P(Valencia gana) : {res_B.prob_b_gana*100:.1f}%")

fig_mc = chart_monte_carlo(
    df_iters_A, df_iters_B, res_A, res_B,
    cand_b_name_a="Abelardo de la Espriella",
    cand_b_name_b="Paloma Valencia",
)
FIGURAS12["12_02_monte_carlo"] = fig_mc
fig_mc.show()


⏳ Corriendo simulaciones Monte Carlo (20,000 iter. × 2 escenarios, seed=42)…

📊 Escenario A — Iván Cepeda vs Abelardo de la Espriella
   Cepeda    : 49.3%  IC80: 48.4–50.2%
   Espriella : 50.7%  IC80: 49.8–51.6%
   P(Cepeda gana)    : 17.5%
   P(Espriella gana) : 82.5%
   P(Empate técnico <2pp): 66.9%

📊 Escenario B — Iván Cepeda vs Paloma Valencia
   Cepeda  : 49.4%  IC80: 48.3–50.4%
   Valencia: 50.6%  IC80: 49.6–51.7%
   P(Cepeda gana)   : 20.9%
   P(Valencia gana) : 79.1%


### 12.3 — Análisis de sensibilidad: tres swing factors

Los tres factores que, según el documento forense, tienen mayor capacidad de cambiar el resultado. El eje X muestra el valor del parámetro; el eje Y el % de voto proyectado en 2V.

In [73]:
# ── 12.3 Análisis de sensibilidad — swing factors ─────────────────────
# Los 3 swing factors (nombre, rango, n_puntos, n_iter_mc) están centralizados
# en ``encuestas_lib.viz.charts.step12.SWING_FACTORS_DOC``.
# Para tunearlos: editar SOLO ese módulo.
from encuestas_lib.viz.charts.step12 import SWING_FACTORS_DOC, chart_sensibilidad

print(f"⏳ Computando análisis de sensibilidad ({len(SWING_FACTORS_DOC)} swing factors)…")

# chart_sensibilidad espera exactamente 3 DataFrames (subplot_titles hardcoded).
# Si quieres más/menos factores, edita también la función en step12.py.
if len(SWING_FACTORS_DOC) != 3:
    raise ValueError(
        f"SWING_FACTORS_DOC tiene {len(SWING_FACTORS_DOC)} elementos; "
        "chart_sensibilidad requiere exactamente 3.  Edita step12.py."
    )

dfs_sens = []
for sf in SWING_FACTORS_DOC:
    df_s = sensibilidad_2v(
        PESOS_PV_DOC, MATRIZ_A,
        candidato_a="Iván Cepeda", candidato_b="Abelardo de la Espriella",
        param_name=sf.nombre,
        param_pv_cand=sf.pv_cand, param_destino=sf.destino,
        rango_param=(sf.rango_min, sf.rango_max),
        n_puntos=sf.n_puntos, n_iter_mc=sf.n_iter_mc,
    )
    dfs_sens.append(df_s)

fig_sens = chart_sensibilidad(*dfs_sens)
FIGURAS12["12_03_sensibilidad"] = fig_sens
fig_sens.show()


⏳ Computando análisis de sensibilidad (3 swing factors)…


### 12.4 — Polymarket vs encuestas vs modelo MC

In [74]:
# ── 12.4 Polymarket vs encuestas vs modelo de transferencia ─────────
# Datos del documento forense (cifras del 16-17 may) centralizados en
# ``encuestas_lib.viz.charts.step12.POLYMARKET_SNAPSHOT_DOC``.
# Para actualizar snapshots: editar SOLO ese módulo.
from encuestas_lib.viz.charts.step12 import (
    chart_polymarket,
    construir_comparativo_polymarket,
)

COMPARATIVO = construir_comparativo_polymarket(res_A, res_B)

fig_pm = chart_polymarket(COMPARATIVO)
FIGURAS12["12_04_polymarket"] = fig_pm
fig_pm.show()


### 12.5 — Techo de rechazo por candidato

Valencia tiene el menor techo de rechazo (15–17%) — su ventaja estructural en 2V.

In [75]:
# ── 12.5 Techo de rechazo por candidato ──────────────────────────────
# Rangos del documento forense centralizados en
# ``encuestas_lib.viz.charts.step12.TECHO_RECHAZO_DOC``.
# Para tunearlos: editar SOLO ese módulo.
from encuestas_lib.analysis.weighting import resolve_weights
from encuestas_lib.viz.charts.step12 import TECHO_RECHAZO_DOC, chart_techo_rechazo

t_rec = calcular_techo_rechazo(
    df, list(TECHO_RECHAZO_DOC.keys()),
    resolve_weights(config.weighting, config.surveys),
)
print("Techo de rechazo estimado desde microdatos:")
print(t_rec.to_string(index=False))

fig_tr = chart_techo_rechazo(t_rec, TECHO_RECHAZO_DOC)
FIGURAS12["12_05_techo_rechazo"] = fig_tr
fig_tr.show()


Techo de rechazo estimado desde microdatos:
               candidato  pct_rechazo   metodo
             Iván Cepeda          NaN sin_dato
Abelardo de la Espriella          NaN sin_dato
         Paloma Valencia          NaN sin_dato


### 12.6 — Geografía del petrismo

In [76]:
# ── 12.6 Voto Cepeda por región vs aprobación Petro ─────────────────
from encuestas_lib.viz.charts.step12 import chart_geografia_petrismo

fig_reg = chart_geografia_petrismo(tablas)
FIGURAS12["12_06_geografia_petrismo"] = fig_reg
fig_reg.show()


### 12.7 — Voto joven y abstención diferencial

La ruptura generacional (Cepeda 2:1 en 18-34) y la concentración de indecisos en ese grupo son el swing factor principal para la segunda vuelta.

In [77]:
# ── 12.7 Abstención diferencial y voto joven ─────────────────────────
from encuestas_lib.viz.charts.step12 import chart_voto_joven

fig_abs = chart_voto_joven(tablas)
FIGURAS12["12_07_voto_joven"] = fig_abs
fig_abs.show()


### 12.8 — Panel ejecutivo: escenarios y probabilidades

In [78]:
# ── 12.8 Panel ejecutivo: escenarios finales ─────────────────────────
# Escenarios y probabilidades del documento forense centralizados en
# ``encuestas_lib.viz.charts.step12.ESCENARIOS_DOC``.
# Para tunearlos: editar SOLO ese módulo.
from encuestas_lib.viz.charts.step12 import (
    chart_panel_ejecutivo,
    construir_escenarios_consolidados,
)

ESCENARIOS = construir_escenarios_consolidados(res_A, res_B)

fig_ej = chart_panel_ejecutivo(ESCENARIOS)
FIGURAS12["12_08_panel_ejecutivo"] = fig_ej
fig_ej.show()

print("\n📋 Resumen de probabilidades consolidadas:")
print(ESCENARIOS.to_string(index=False))



📋 Resumen de probabilidades consolidadas:
                     escenario  prob_doc  prob_modelo          tipo
             1V: Cepeda lidera      87.0         85.0            1V
            Espriella 2° lugar      70.0         68.0            1V
             Valencia 3° lugar      63.0         62.0            1V
      2V: Cepeda gana (esc. A)      48.5         18.0            2V
   2V: Espriella gana (esc. A)      40.0         82.0            2V
    2V: Valencia gana (esc. B)      50.0         79.0            2V
Empate técnico 2V (|dif| <2pp)      35.0         67.0 Incertidumbre


### 12.9 — Exportar Step 12 al dashboard

Actualiza `dashboard_interactivo.html` con todas las gráficas del Step 12.

In [79]:
# ── 12.9 Exportar todas las figuras (Step 11 + Step 12) al dashboard ─
from encuestas_lib.viz.dashboard import (
    SECCIONES_STEP11, SECCIONES_STEP12, export_dashboard,
)

# Combinar Step 11 + Step 12 (idempotente, sobrescribe el archivo HTML)
try:
    todas_figuras = {**FIGURAS, **FIGURAS12}
    secciones_11 = SECCIONES_STEP11
except NameError:
    # Step 11 no se corrió en esta sesión: solo Step 12
    todas_figuras = dict(FIGURAS12)
    secciones_11 = ()

out_path = export_dashboard(
    figuras=todas_figuras,
    out_path=FIG_DIR / "dashboard_interactivo.html",
    secciones_step11=secciones_11,
    secciones_step12=SECCIONES_STEP12,
)
size_mb = out_path.stat().st_size / 1e6

print(f"✅ Dashboard actualizado: {out_path.name} ({size_mb:.2f} MB)")
try:
    print(f"   Step 11: {len(FIGURAS)} gráficas  |  Step 12: {len(FIGURAS12)} gráficas")
except NameError:
    print(f"   Step 12: {len(FIGURAS12)} gráficas (Step 11 no disponible)")


✅ Dashboard actualizado: dashboard_interactivo.html (0.77 MB)
   Step 11: 13 gráficas  |  Step 12: 9 gráficas


---

## 🎉 Pipeline terminado

Si llegaste aquí sin errores, tienes:

- ✅ Encuestas armonizadas en un único DataFrame (`df`)
- ✅ ~25 tablas analíticas en `tablas`, exportadas a Excel y JSON en Drive
- ✅ Auditoría forense que confirma que las tablas cierran a 100%
- ✅ **Dashboard interactivo** en `data/outputs/graficas/dashboard_interactivo.html`

### Próximos pasos

1. **Nueva encuesta**: edita `configs/surveys.yaml`, sube el Excel a `data/raw/`. El pipeline detecta el cambio automáticamente (fingerprint) y re-ingesta.
2. **Nueva encuestadora**: crea un Reader en `encuestas_lib/readers/`. Ver `README.md` § 'Agregar una nueva encuesta'.
3. **Actualizar dashboard**: vuelve a correr el Step 11 — el dashboard refleja los datos más recientes.
